## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [49]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.documents import Document

import gradio as gr

In [50]:
MODEL = "gemini-2.5-flash-lite"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [5]:
retriever = vectorstore.as_retriever()
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

### These LangChain objects implement the method `invoke()`

In [6]:
retriever.invoke("Who is Avery?")

[Document(id='a2753f68-c1d9-4efa-b7a1-c6a8c4ebd497', metadata={'doc_type': 'employees', 'source': 'C:\\Users\\Máté\\Desktop\\Learning\\AI_Engineer_Core\\llm_engineering\\week5\\knowledge-base\\employees\\Avery Lancaster.md'}, page_content='- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.'),
 Document(id='a1b80d8a-3913-461a-8fee-9b07e2372f09', metadata={'source': 'C:\\Users\\Máté\\Desktop\\Learning\\AI_Engineer_Core\\llm_engineering\\week5\\knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2021**: **Exceptional**  \n  Avery's decisive transition to remote work and rapid adoption of digital tools led to record-high customer sati

In [7]:
llm.invoke("Who is Avery?")

AIMessage(content='"Avery" is a name that can refer to many different people, and without more context, it\'s impossible to know who you\'re asking about.\n\nTo help me figure out who you mean, could you provide more information? For example:\n\n*   **What is Avery known for?** (e.g., an actor, a musician, a historical figure, a character in a book/movie, a friend, a colleague, etc.)\n*   **Where did you hear about Avery?** (e.g., a specific article, a TV show, a conversation, a company, etc.)\n*   **Do you have any other details about them?** (e.g., their profession, their nationality, their approximate age, etc.)\n\nOnce you give me a little more to go on, I can try to identify the Avery you\'re interested in!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5379-994c-7191-ab56-0a16a0a090b5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_

## Time to put this together!

In [8]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [42]:
SYSTEM_PROMPT_SEARCH = """
You are a helpful assistant that focuses on formatting user questions for retrieval.
You are given a question and the conversation history.
Your task is to combine the question and the conversation history into a single string
that can be used for retrieval.
The conversation history might not be relevant to the latest question, so only include
the parts of the conversation that are relevant to the question.
If the conversation history is not relevant, just return the question.
Only return the combined string; do not include any explanations or formatting.
"""

In [15]:
def fetch_context(question: str) -> list[Document]:
    """
    Retrieve relevant context documents for a question.
    """
    return retriever.invoke(question, k=5)


In [43]:
def create_search_query(question: str, history: list[dict] = []) -> str:
    """
    Create a search query for retrieval based on the question and conversation history.
    """
    prior_user_messages = "\n".join(m[0] for m in history)
    
    messages = [
        SystemMessage(content=SYSTEM_PROMPT_SEARCH),
        # Add the context of what was asked before
        HumanMessage(content=f"Previous questions: {prior_user_messages}"),
        # Add the current question
        HumanMessage(content=f"Current question: {question}")
    ]
    response = llm.invoke(messages)
    return response.content

In [51]:
def convert_to_messages(history):
    messages = []
    for user_msg, bot_msg in history:
        messages.append(HumanMessage(content=user_msg))
        messages.append(AIMessage(content=bot_msg))
    return messages

In [52]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list[Document]]:
    """
    Answer the given question with RAG; return the answer and the context documents.
    """
    query = create_search_query(question, history)
    docs = fetch_context(query)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content

In [53]:
create_search_query("Who is Avery?", [])

'Who is Avery?'

In [54]:
answer_question("Who is Averi Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She co-founded the company in 2015 and has been instrumental in its growth and success as a leading Insurance Tech provider. Avery is recognized for her innovative leadership and expertise in risk management.'

## What could possibly come next? 😂

In [55]:
gr.ChatInterface(answer_question).launch()

c:\Users\Máté\Desktop\Learning\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!